In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, optimal_leaf_ordering
from scipy.spatial.distance import squareform
import seaborn as sns
from typing import Dict, Any

In [ ]:
# Helper function to turn our crossplay_results into a matrix

def reshape_nested_dict_to_matrix(data: Dict[str, Dict[str, np.ndarray]]) -> tuple[np.ndarray, list[str]]:
    """
    Reshape nested dictionary data into one large matrix combining all algorithms.
    
    Args:
        data: Dictionary with structure {ALGORITHM: {robot_uuid: array[1:n_humans]}}
    
    Returns:
        tuple: (combined_matrix, row_labels) where:
        - combined_matrix: Matrix with all robots from all algorithms
          - Rows: Each robot instance (algorithm_robot_uuid combinations)
          - Columns: human_id (0 to n_humans-1)
        - row_labels: List of strings identifying each row as "algorithm_robot_uuid"
    """
    all_rows = []
    row_labels = []
    
    # Sort algorithms for consistent ordering
    algorithms = sorted(data.keys())
    
    for algorithm in algorithms:
        robot_data = data[algorithm]
        
        robot_uuids = robot_data.keys()
        
        for robot_uuid in robot_uuids:
            robot_array = robot_data[robot_uuid]
            all_rows.append(robot_array)
            row_labels.append(f"{algorithm}_{robot_uuid}")
        
        print(f"Algorithm '{algorithm}': Added {len(robot_uuids)} robots")
    
    # Stack all rows vertically to create the combined matrix
    combined_matrix = np.vstack(all_rows)
    
    print(f"\nCombined matrix shape: {combined_matrix.shape} "
          f"({len(row_labels)} total robots x {combined_matrix.shape[1]} humans)")
    
    return combined_matrix, row_labels

In [ ]:
xp_r = np.load('/home/leo/assistive-autonomy-github/assistax/assistax/outputs/crossplay/scratchitch/2025-09-01/12-42-33/crossplay_test_results.npy', allow_pickle=True).item()
xp_matrix, row_labels = reshape_nested_dict_to_matrix(xp_r)

print("XP Matrix Shape:", xp_matrix.shape)

In [ ]:
ENV = "scratchitch"  # Define ENV since it's used in the title
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(xp_matrix, aspect='equal', cmap='viridis', extent=[0, 10, 0, 10], origin='lower')
ax.set_xlabel("Human")
ax.set_ylabel("Robot")
ax.set_title(ENV)
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Sort with clustering algorithm

